# Imports

In [1]:
import pickle
import numpy as np
import pandas as pd
import re

Load files and create features

In [2]:
# Load saved model files
with open("preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

with open("voting_ensemble.pkl", "rb") as f:
    voting_ensemble = pickle.load(f)

with open("author_counts.pkl", "rb") as f:
    author_counts = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

# feature engineering

# Target Audience
def get_target_audience(text):
    if any(w in text for w in ["дитина", "казка", "малюк", "абетка", "школа", "дитячий", "вірш"]):
        return "Children"
    if any(w in text for w in ["молодь", "підліток", "зрілість", "дорослішання"]):
        return "Teenagers"
    if any(w in text for w in ["подружжя", "шлюб", "виховання", "родина", "батьки", "мати"]):
        return "Family"
    if any(w in text for w in ["душа", "молитва", "покаяння", "таїнство", "піст", "богослов", "служіння"]):
        return "Adults"
    return "General"

# Biblical Reference Level
def classify_biblical_level(text):
    high_roots = [
        "біблі", "євангел", "святе письмо", "псалм", "псалтир", "пророк", "апостол",
        "тлумачен", "коментар", "старий завіт", "новий завіт", "богослов", "юдей", "слово боже"
    ]
    medium_roots = [
        "притч", "роздум", "заповід", "святий", "катехизм", "духовн", "повчан", "доктрин", "догма"
    ]
    low_roots = [
        "мораль", "мудр", "цінн", "щаст", "краса", "любов", "натхнен", "емоці", "розвиток", "поради"
    ]
    extra_high = ["господь", "ісус", "христос", "спаситель", "учні", "воскресіння", "сповідь"]
    extra_medium = ["чесноти", "світогляд", "християнськ", "віровчення"]

    high_score = sum(1 for root in high_roots if re.search(root, text))
    medium_score = sum(1 for root in medium_roots if re.search(root, text))
    low_score = sum(1 for root in low_roots if re.search(root, text))

    high_score += sum(1 for i in extra_high if i in text)
    medium_score += sum(1 for i in extra_medium if i in text)

    if high_score >= max(medium_score, low_score) and high_score > 0:
        return "High"
    elif medium_score >= max(high_score, low_score) and medium_score > 0:
        return "Medium"
    elif low_score > 0:
        return "Low"
    else:
        return "Very Low"

# Application Type
def get_application_type(text):
    if any(w in text for w in ["день", "роздуми"]):
        return "Daily Prayers"
    if any(w in text for w in ["вивчення", "уроки", "курс", "тлумачення"]):
        return "Studies"
    if any(w in text for w in ["біографія", "житія", "історія життя"]):
        return "Biography"
    if any(w in text for w in ["молитви", "молитовник", "прохання", "молебень", "акафісник"]):
        return "Prayers"
    if any(w in text for w in ["казка", "повість", "розповідь"]):
        return "Tales"
    return "General"

# Event Relevance
def get_event_relevance(text):
    if any(w in text for w in ["різдво", "новий рік", "коляда"]):
        return "Christmas"
    if "великдень" in text:
        return "Easter"
    if any(w in text for w in ["піст", "страсті", "страсна"]):
        return "Fasting"
    if any(w in text for w in ["випуск", "школа", "навчання"]):
        return "School Time"
    return "No Event"

# Content Format
def get_content_format(text):
    if any(w in text for w in ["поезія", "вірш", "вірші"]):
        return "Poetry"
    if any(w in text for w in ["оповідання", "повість", "розповідь", "притча"]):
        return "Stories"
    if any(w in text for w in ["молитва", "акафісник", "піснеспів"]):
        return "Prayer"
    if any(w in text for w in ["лист", "діалог"]):
        return "Dialogue/Letters"
    return "Prose"

# Main Prediction Function

In [3]:
def predict_new_book(book_info):
    # Extract features
    title = book_info.get("Title", "")
    desc = book_info.get("Description", "")
    full_text = (title + " " + desc).lower()

    features_dict = {
        "Author": book_info.get("Author", ""),
        "Pages": book_info.get("Pages", np.nan),
        "Price": book_info.get("Price", np.nan),
        "Production_Year": book_info.get("Production_Year", np.nan),
        "Format": book_info.get("Format", ""),
        "Category": book_info.get("Category", ""),
        "Subcategory": book_info.get("Subcategory", ""),
        "Cover Type": book_info.get("Cover Type", ""),
        "Друкарня": book_info.get("Друкарня", ""),
        "Target Audience": get_target_audience(full_text),
        "Biblical Level": classify_biblical_level(full_text),
        "Application Type": get_application_type(full_text),
        "Event Relevance": get_event_relevance(full_text),
        "Content Format": get_content_format(full_text)
    }

    features_dict["Is_Expensive"] = int(features_dict["Price"] > 250)
    features_dict["Price_per_Page"] = features_dict["Price"] / (features_dict["Pages"] + 1)
    features_dict["Pages_Category"] = pd.cut([features_dict["Pages"]], bins=[0, 100, 300, 1000], labels=["Small", "Medium", "Big"])[0]
    features_dict["Price_Category"] = pd.cut([features_dict["Price"]], bins=[0, 150, 400, 10000], labels=["Cheap", "Normal", "Expensive"])[0]

    # Add Author Score
    features_dict["Author Score"] = author_counts.get(features_dict["Author"], 0)

    # Prepare input dataset
    input_df = pd.DataFrame([features_dict])

    # Drop 'Author' column (not used for prediction)
    input_df.drop(columns=["Author"], inplace=True)

    # Preprocess
    X_input = preprocessor.transform(input_df)

    # Predict
    probs = voting_ensemble.predict_proba(X_input)[0]
    pred_class_idx = np.argmax(probs)
    pred_class = label_encoder.inverse_transform([pred_class_idx])[0]

    print(f"\nPredicted Class: {pred_class}")
    for class_name, prob in zip(label_encoder.classes_, probs):
        print(f"{class_name}: {prob:.2%}")

    return pred_class, probs

Check results

In [4]:
new_book = {
    "Title": "Ісус навчає мудрості",
    "Author": "Холева Марцін",
    "Pages": 48,
    "Price": 30,
    "Production_Year": 2019,
    "Format": "103х145",
    "Category": "Богослужбові книги",
    "Subcategory": "Молитовники та акафісники",
    "Cover Type": "м'яка",
    "Description": "Йдучи слідами Ісусового хреста, ми крокуємо шляхом, який нам вказує Божа мудрість. Мудрий Бог хоче, щоб ми були мудрими у своєму житті та йшли дорогою мудрості.Чотирнадцять стоянь Хресної дороги - це чотирнадцять слів Бога про мудрість, слів, з якими Бог звертається до нас.",
    "Друкарня": "ПП „Слово””"
}

predict_new_book(new_book)


Predicted Class: High
High: 81.38%
Low: 18.62%


('High', array([0.8137536, 0.1862464]))